# Ejercicio · Point Transformer sobre Teeth3DS+ — segmentación de dientes (FDI)

Reproducción del ejemplo de entrenamiento de una **librería PyTorch para nubes de puntos**,
midiendo el **tiempo de entrenamiento** y entendiendo **los datos** (cuántos hay, qué es test).

- **Librería:** [PyTorch Geometric (PyG)](https://pytorch-geometric.readthedocs.io) `2.8` + `pyg-lib`
- **Modelo:** Point Transformer (segmentación) — [ejemplo oficial](https://github.com/pyg-team/pytorch_geometric/blob/master/examples/point_transformer_segmentation.py)
- **Datos:** **Teeth3DS+ local** (`data/raw/teeth3ds`) en vez de ShapeNet — misma tarea
  (segmentar partes de una nube), pero las "partes" son **dientes (FDI)**. Cero descarga.
- **Kernel:** `Dental GPU (3DGS)` (torch cu128 + RTX 5070)

> Es un **prototipo directo del `segmentation-agent`**: segmentar la nube de puntos por diente FDI.
> Reproduce el modelo/bucle oficial; solo cambia el *loader* (mallas `.obj` + labels `.json` → PyG)
> e IoU calculado a mano (la firma de `torchmetrics.jaccard_index` cambió en 1.x).

### 0 · Entorno

In [1]:
import torch, torch_geometric, pyg_lib
print("torch          :", torch.__version__)
print("torch_geometric:", torch_geometric.__version__, "| pyg_lib:", pyg_lib.__version__)
print("CUDA           :", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

torch          : 2.11.0+cu128
torch_geometric: 2.8.0 | pyg_lib: 0.7.0+pt211cu128
CUDA           : True | NVIDIA GeForce RTX 5070


### 1 · Datos — Teeth3DS+ local (¿qué cantidad tenemos y qué es test?)

Cada caso = una arcada: malla `.obj` (~76k–168k vértices con color por vértice) + `.json` con
una etiqueta **FDI por vértice** (`0` = encía). Submuestreamos a 2048 puntos/muestra y hacemos
split **por paciente** (sin fuga: un paciente no está a la vez en train y test).

In [2]:
import glob, json, os, os.path as osp, time
from pathlib import Path
import numpy as np
import vtk
from vtk.util.numpy_support import vtk_to_numpy
import torch_geometric.transforms as T
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

vtk.vtkObject.GlobalWarningDisplayOff()  # los .obj traen color por vértice -> flood de avisos

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "data/raw/teeth3ds").exists():
    ROOT = ROOT.parent
MESH = ROOT / "data/raw/teeth3ds/3D_scans_per_patient_obj_files"
LAB = ROOT / "data/raw/teeth3ds/ground-truth_labels_instances"
NUM_POINTS, BATCH = 2048, 4


def read_case(obj_path, json_path):
    r = vtk.vtkOBJReader(); r.SetFileName(str(obj_path)); r.Update()
    pos = vtk_to_numpy(r.GetOutput().GetPoints().GetData()).astype(np.float32)
    lab = np.asarray(json.load(open(json_path))["labels"], dtype=np.int64)
    assert len(lab) == pos.shape[0]
    return pos, lab


t0 = time.perf_counter()
patients = sorted(os.listdir(MESH))
raw = []
for pid in patients:
    for jaw in ("lower", "upper"):
        op, jp = MESH / pid / f"{pid}_{jaw}.obj", LAB / pid / f"{pid}_{jaw}.json"
        if op.exists() and jp.exists():
            raw.append((pid, *read_case(op, jp)))

all_codes = sorted({int(x) for _, _, lab in raw for x in np.unique(lab)})
code2idx = {c: i for i, c in enumerate(all_codes)}   # 0(encía)->0, FDI->1..K-1
K = len(all_codes)
verts = [p.shape[0] for _, p, _ in raw]
print(f"{len(raw)} mallas de {len(patients)} pacientes en {time.perf_counter()-t0:.1f}s")
print(f"clases (encía+FDI): {K}  ->  {all_codes}")
print(f"vértices/malla: min={min(verts)} media={sum(verts)//len(verts)} max={max(verts)} -> submuestreo a {NUM_POINTS}")

600 mallas de 300 pacientes en 41.1s
clases (encía+FDI): 32  ->  [0, 11, 12, 13, 14, 15, 16, 17, 21, 22, 23, 24, 25, 26, 27, 28, 31, 32, 33, 34, 35, 36, 37, 38, 41, 42, 43, 44, 45, 46, 47, 48]
vértices/malla: min=13034 media=116527 max=219518 -> submuestreo a 2048


In [3]:
norm = T.NormalizeScale()
data_all = []
for pid, pos, lab in raw:
    y = torch.tensor([code2idx[int(x)] for x in lab], dtype=torch.long)
    d = norm(Data(pos=torch.from_numpy(pos), y=y)); d.pid = pid
    data_all.append(d)
raw.clear()  # libera labels crudos + tuplas (pos ya vive en data_all)

# Split REAL 80/20 POR PACIENTE (antes: solo 2 pacientes de test -> overfitting
# no medible). Semilla fija = reproducible; sin fuga (un paciente entero cae en
# train O en test, nunca en ambos). Con el Teeth3DS+ completo (300 pacientes)
# esto pasa de 20/4 mallas a ~480/120 -> el val set ya es representativo.
ids = sorted({d.pid for d in data_all})
rng = np.random.default_rng(42); rng.shuffle(ids)
n_test = max(1, round(0.20 * len(ids)))
test_patients = set(ids[:n_test])
train_list = [d for d in data_all if d.pid not in test_patients]
test_list = [d for d in data_all if d.pid in test_patients]
print(f"pacientes: {len(ids)}  ->  train {len(ids) - n_test} / test {n_test}  (80/20, seed 42)")
print(f"train = {len(train_list)} mallas   <- entrenamiento")
print(f"test  = {len(test_list)} mallas    <- evaluación")

sampler = T.FixedPoints(NUM_POINTS, replace=False, allow_duplicates=True)


class TeethDS:
    def __init__(self, items): self.items = items
    def __len__(self): return len(self.items)
    def __getitem__(self, i): return sampler(self.items[i].clone())  # submuestreo aleatorio/época
    num_classes = K


train_loader = DataLoader(TeethDS(train_list), batch_size=BATCH, shuffle=True)
test_loader = DataLoader(TeethDS(test_list), batch_size=BATCH, shuffle=False)
print("num_classes:", K)

pacientes: 300  ->  train 240 / test 60  (80/20, seed 42)
train = 480 mallas   <- entrenamiento
test  = 120 mallas    <- evaluación
num_classes: 32


### 2 · Modelo — Point Transformer (segmentación, *pos-only*)

In [4]:
import torch
import torch.nn.functional as F
from torch.nn import Linear as Lin
from torch_geometric.nn import MLP, PointTransformerConv, fps, knn, knn_graph, knn_interpolate
from torch_geometric.utils import scatter


class TransformerBlock(torch.nn.Module):
    def __init__(self, ic, oc):
        super().__init__()
        self.lin_in = Lin(ic, ic); self.lin_out = Lin(oc, oc)
        self.pos_nn = MLP([3, 64, oc], norm=None, plain_last=False)
        self.attn_nn = MLP([oc, 64, oc], norm=None, plain_last=False)
        self.transformer = PointTransformerConv(ic, oc, pos_nn=self.pos_nn, attn_nn=self.attn_nn)

    def forward(self, x, pos, ei):
        x = self.lin_in(x).relu(); x = self.transformer(x, pos, ei); return self.lin_out(x).relu()


class TransitionDown(torch.nn.Module):
    """Submuestrea (FPS) y agrupa features por kNN (max-pool)."""
    def __init__(self, ic, oc, ratio=0.25, k=16):
        super().__init__()
        self.k, self.ratio = k, ratio; self.mlp = MLP([ic, oc], plain_last=False)

    def forward(self, x, pos, batch):
        idc = fps(pos, ratio=self.ratio, batch=batch)
        sb = batch[idc] if batch is not None else None
        idk = knn(pos, pos[idc], k=self.k, batch_x=batch, batch_y=sb)
        x = self.mlp(x)
        xo = scatter(x[idk[1]], idk[0], dim=0, dim_size=idc.size(0), reduce="max")
        return xo, pos[idc], sb


class TransitionUp(torch.nn.Module):
    """Interpola features de baja resolución de vuelta a alta."""
    def __init__(self, ic, oc):
        super().__init__()
        self.mlp_sub = MLP([ic, oc], plain_last=False); self.mlp = MLP([oc, oc], plain_last=False)

    def forward(self, x, x_sub, pos, pos_sub, batch=None, batch_sub=None):
        x_sub = self.mlp_sub(x_sub)
        xi = knn_interpolate(x_sub, pos_sub, pos, k=3, batch_x=batch_sub, batch_y=batch)
        return self.mlp(x) + xi


class Net(torch.nn.Module):
    def __init__(self, ic, oc, dim_model, k=16):
        super().__init__()
        self.k = k; ic = max(ic, 1)
        self.mlp_input = MLP([ic, dim_model[0]], plain_last=False)
        self.transformer_input = TransformerBlock(dim_model[0], dim_model[0])
        self.transformers_up = torch.nn.ModuleList(); self.transformers_down = torch.nn.ModuleList()
        self.transition_up = torch.nn.ModuleList(); self.transition_down = torch.nn.ModuleList()
        for i in range(len(dim_model) - 1):
            self.transition_down.append(TransitionDown(dim_model[i], dim_model[i + 1], k=k))
            self.transformers_down.append(TransformerBlock(dim_model[i + 1], dim_model[i + 1]))
            self.transition_up.append(TransitionUp(dim_model[i + 1], dim_model[i]))
            self.transformers_up.append(TransformerBlock(dim_model[i], dim_model[i]))
        self.mlp_summit = MLP([dim_model[-1], dim_model[-1]], norm=None, plain_last=False)
        self.transformer_summit = TransformerBlock(dim_model[-1], dim_model[-1])
        self.mlp_output = MLP([dim_model[0], 64, oc], norm=None)

    def forward(self, x, pos, batch=None):
        if x is None:
            x = torch.ones((pos.shape[0], 1), device=pos.device)
        ox, op, ob = [], [], []
        x = self.mlp_input(x); ei = knn_graph(pos, k=self.k, batch=batch)
        x = self.transformer_input(x, pos, ei); ox.append(x); op.append(pos); ob.append(batch)
        for i in range(len(self.transformers_down)):
            x, pos, batch = self.transition_down[i](x, pos, batch=batch)
            ei = knn_graph(pos, k=self.k, batch=batch); x = self.transformers_down[i](x, pos, ei)
            ox.append(x); op.append(pos); ob.append(batch)
        x = self.mlp_summit(x); ei = knn_graph(pos, k=self.k, batch=batch)
        x = self.transformer_summit(x, pos, ei)
        for i in range(len(self.transformers_down)):
            x = self.transition_up[-i - 1](x=ox[-i - 2], x_sub=x, pos=op[-i - 2],
                                           pos_sub=op[-i - 1], batch_sub=ob[-i - 1], batch=ob[-i - 2])
            ei = knn_graph(op[-i - 2], k=self.k, batch=ob[-i - 2])
            x = self.transformers_up[-i - 1](x, op[-i - 2], ei)
        return F.log_softmax(self.mlp_output(x), dim=-1)

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Net(1, K, dim_model=[32, 64, 128, 256, 512], k=16).to(device)  # x=None -> solo geometría
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)
print(f"Point Transformer seg · {sum(p.numel() for p in model.parameters())/1e6:.2f}M params · {device}")

Point Transformer seg · 4.59M params · cuda


### 3 · Entrenamiento con loss ponderada por clase + medición de tiempo

**Ajustar la loss por clase.** La versión *pos-only* sin
pesos colapsaba a la clase mayoritaria (**encía**, ~40% de los puntos): `train_acc` clavado en ~0.42
y **cero dientes segmentados**. Ahora la loss usa **pesos por frecuencia inversa** (*median-frequency
balancing*): las clases raras (cada diente) pesan hasta ~50× más que la encía, así el modelo no puede
«ganar» ignorándolas.

Diagnóstico añadido: **`tooth_acc`** = acierto SOLO en los puntos de diente (no-encía). Es el número
honesto — el `mIoU` está inflado por la convención «parte ausente → 1.0». Si `tooth_acc` sube desde
~0, la loss ponderada **rompió el colapso** (aunque `acc` global puede bajar: el modelo deja de
acertar «gratis» prediciendo encía).

In [6]:
import time

# --- Pesos por clase: el arreglo del comentario del jefe (#1) ----------------
# Sin pesos, la ENCÍA (clase 0, ~40% de los puntos) domina el gradiente y el
# modelo colapsa prediciéndola en todo. Ponderamos por frecuencia inversa
# (median-frequency balancing): peso_c = mediana(freq)/freq_c. Los dientes raros
# pesan más; clases ausentes en train -> peso 0 (si no, sería infinito).
counts = torch.zeros(K)
for d in train_list:
    counts += torch.bincount(d.y, minlength=K).float()
freq = counts / counts.sum()
present = counts > 0
weights = torch.zeros(K)
weights[present] = (freq[present].median() / freq[present]).clamp(max=50.0)
W = weights.to(device)
ratio = (weights[1:][present[1:]].mean() / weights[0]).item()
print(f"encía (clase 0): {freq[0] * 100:.1f}% de los puntos · peso {weights[0]:.3f}")
print(f"dientes: peso medio {weights[1:][present[1:]].mean():.2f} · máx {weights.max():.2f} "
      f"(un diente pesa ~{ratio:.0f}x más que la encía en la loss)")


def train():
    model.train(); tl = correct = total = tcorr = ttot = 0
    for data in train_loader:
        data = data.to(device); optimizer.zero_grad()
        out = model(None, data.pos, data.batch)
        loss = F.nll_loss(out, data.y, weight=W)      # <-- loss ponderada por clase
        loss.backward(); optimizer.step()
        tl += loss.item() * data.num_graphs
        pred = out.argmax(1)
        correct += pred.eq(data.y).sum().item(); total += data.num_nodes
        tm = data.y != 0                              # solo dientes (foreground)
        tcorr += pred[tm].eq(data.y[tm]).sum().item(); ttot += int(tm.sum())
    return tl / len(train_loader.dataset), correct / total, tcorr / max(ttot, 1)


@torch.no_grad()
def test(loader):
    model.eval(); parts = list(range(K)); ious = []; tcorr = ttot = 0
    for data in loader:
        data = data.to(device); pred = model(None, data.pos, data.batch).argmax(1)
        tm = data.y != 0
        tcorr += pred[tm].eq(data.y[tm]).sum().item(); ttot += int(tm.sum())
        sizes = (data.ptr[1:] - data.ptr[:-1]).tolist()
        for p, y in zip(pred.split(sizes), data.y.split(sizes)):
            pi = []
            for cc in parts:
                a, b = (p == cc), (y == cc); u = (a | b).sum().item()
                pi.append(1.0 if u == 0 else (a & b).sum().item() / u)
            ious.append(sum(pi) / len(pi))
    return sum(ious) / len(ious), tcorr / max(ttot, 1)


EPOCHS = 20
ets = []; t0 = time.perf_counter()
for epoch in range(1, EPOCHS + 1):
    if device.type == "cuda": torch.cuda.synchronize()
    te = time.perf_counter(); loss, acc, tacc = train()
    if device.type == "cuda": torch.cuda.synchronize()
    dt = time.perf_counter() - te; ets.append(dt)
    iou, ttacc = test(test_loader)
    print(f"época {epoch:02d} · loss {loss:.3f} · acc {acc:.3f} · "
          f"tooth_acc train {tacc:.3f} / test {ttacc:.3f} · mIoU {iou:.3f} · {dt:.2f}s")
    scheduler.step()

avg = sum(ets) / len(ets)
print(f"\ntiempo total {time.perf_counter() - t0:.1f}s · media/época {avg:.2f}s")
print("tooth_acc = acierto SOLO en puntos de diente (no-encía): revela si segmenta "
      "dientes o solo pinta encía. Antes (sin pesos) era ~0 por el colapso.")

encía (clase 0): 44.0% de los puntos · peso 0.038
dientes: peso medio 5.67 · máx 50.00 (un diente pesa ~147x más que la encía en la loss)


/home/lgarbayo/.venvs/dental-gpu/lib/python3.13/site-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(


época 01 · loss 2.883 · acc 0.097 · tooth_acc train 0.157 / test 0.026 · mIoU 0.456 · 4.67s


época 02 · loss 1.880 · acc 0.242 · tooth_acc train 0.322 / test 0.039 · mIoU 0.493 · 4.44s


época 03 · loss 1.534 · acc 0.363 · tooth_acc train 0.442 / test 0.028 · mIoU 0.495 · 4.45s


...

época 16 · loss 0.647 · acc 0.660 · tooth_acc train 0.769 / test 0.027 · mIoU 0.550 · 4.46s


época 17 · loss 0.587 · acc 0.678 · tooth_acc train 0.793 / test 0.029 · mIoU 0.435 · 4.48s


época 18 · loss 0.604 · acc 0.677 · tooth_acc train 0.790 / test 0.025 · mIoU 0.370 · 4.48s


época 19 · loss 0.583 · acc 0.685 · tooth_acc train 0.793 / test 0.027 · mIoU 0.525 · 4.44s


época 20 · loss 0.570 · acc 0.695 · tooth_acc train 0.810 / test 0.055 · mIoU 0.513 · 4.45s

tiempo total 107.8s · media/época 4.46s
tooth_acc = acierto SOLO en puntos de diente (no-encía): revela si segmenta dientes o solo pinta encía. Antes (sin pesos) era ~0 por el colapso.


### 4 · Resumen

**Experimento.** Segmentación de dientes (FDI) **por punto** sobre **Teeth3DS+ completo** con un
Point Transformer *pos-only* (solo geometría XYZ). Es un prototipo directo del `segmentation-agent`.

**Datos.** 600 mallas / 300 pacientes · 32 clases (encía + 31 FDI) · ~117k vértices/malla
(submuestreados a 2048) · split **por paciente** 240 train / 60 test (80/20, seed 42, sin fuga).

**Modelo.** Point Transformer de segmentación (~4.6M parámetros), *pos-only* (sin features de entrada).

**Loss ponderada por clase.** La encía (clase 0, **44%** de los puntos) domina el gradiente; con
*median-frequency balancing* (peso = mediana(freq)/freq, topado a 50) un diente pesa **~147×** más
que la encía, así el modelo no puede «ganar» prediciendo encía. Diagnóstico **`tooth_acc`** = acierto
SOLO en puntos de diente (el `mIoU` queda inflado por la convención «parte ausente → 1.0» y no informa).

**Resultado (20 épocas · RTX 5070 · ~4.5 s/época).**
- El modelo **aprende el train**: `tooth_acc(train)` sube de **0.16 → 0.81** y la loss satura (2.88 → 0.57).
- **No generaliza**: `tooth_acc(test)` se queda **plano en ~0.05** (rango 0.001–0.15), sin tendencia.

**Conclusión.** El cuello de botella **no es la cantidad de datos** — 480 mallas de train bastan para
aprender. Es que **la geometría sola tiene techo de generalización**: un modelo *pos-only* memoriza el
train pero **no identifica el FDI en pacientes nuevos** (simetría izquierda/derecha 11↔21, dientes
vecinos casi idénticos). **La palanca es *features por punto*** (comentario #2 del jefe): añadir el
**gris del CBCT** —o al menos normales— como canal de entrada (`in_channels>1`) para desambiguar lo
que la posición no puede. Hoy bloqueado por falta de CBCT emparejado → vía sintética (issue G).

**Caveats metodológicos.**
- El `tooth_acc(test)` botando es en parte **artefacto de medición**: `FixedPoints` re-submuestrea
  2048 puntos aleatorios *también en test*, así que cada época evalúa sobre un sorteo distinto (alta
  varianza). Un eval determinista (o promediar varios sorteos) lo limpiaría — pero no cerrará el hueco.
- El `mIoU` (~0.4–0.5) es **no informativo** aquí por la convención parte-ausente; por eso miramos `tooth_acc`.

**Conexión con el proyecto.** Este es el esqueleto del **`segmentation-agent`** (entrada nube de puntos,
salida FDI por punto) y la evidencia de por qué necesita el **twin fusionado** (features CBCT sobre la
malla). Validar el FDI contra el informe sería el `fdi-consistency-agent` (ADR 003).